# 04b -- KeepTradeCut Dynasty Rankings

**Purpose:** Scrape KeepTradeCut's dynasty (overall, veteran+rookie) rankings and
load them into the section-04 dynasty-rankings model. KTC embeds the entire player
database as a `var playersArray = [...]` JSON blob in the initial HTML — one GET,
no API/browser needed.

**Two-layer output model** (handles heterogeneous metrics across dynasty sources):
- `fact_dynasty_rankings` — universal **ranking backbone**, one row per
  `snapshot_date x source_name x source_player_id x format`. Columns every source
  shares: `overall_rank`, `positional_rank`, identity, FKs.
- `fact_dynasty_ranking_metrics` — **long** companion, one row per
  `... x metric_key`. Absorbs source-specific metrics (KTC value/tier/trend/crowd/
  market) as `metric_num`/`metric_text` — new sources add ROWS, never columns.
- `dim_dynasty_crosswalk` — `(source, source_player_id)` -> `gsis_id` + `player_key`.

**Formats:** `SF` (`superflexValues`) and `TEPP` (`superflexValues.tepp`). 1QB
omitted (superflex league). RDP (rookie draft picks) excluded — no player identity.

**Outputs:**
- `data/raw/ktc_dynasty_{snapshot_date}.json` — verbatim playersArray (audit/replay)
- `data/fact_dynasty_rankings.parquet`, `data/fact_dynasty_ranking_metrics.parquet`
- `data/dim_dynasty_crosswalk.parquet`; unresolved -> `data/review/review_dynasty_crosswalk.csv`

**Idempotent:** both facts replace-by-`(snapshot_date, source_name)`; crosswalk
replaces `source="KTC"` rows. Run with CWD = repo root.

In [ ]:
# ---- Setup & Config ---------------------------------------------------------
import json
import re
import time
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import pandas as pd
from thefuzz import fuzz, process

# Shared helpers + config (single source of truth)
import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA, _make_session, clean_player_name


@dataclass
class KTCDynastyConfig:
    """KTC dynasty-rankings scrape + load config."""
    url: str = "https://keeptradecut.com/dynasty-rankings"
    source_name: str = "KTC"
    source_site: str = "KeepTradeCut"
    # format -> path into the player's value object (superflex base vs TEP-premium)
    formats: dict = field(default_factory=lambda: {
        "SF":   ("superflexValues",),
        "TEPP": ("superflexValues", "tepp"),
    })
    # per-format metric fields (vary by TEP) read from the format object
    fmt_metric_map: dict = field(default_factory=lambda: {
        "value":           "value",
        "overall_tier":    "overallTier",
        "positional_tier": "positionalTier",
    })
    # format-agnostic metrics (read from superflexValues parent; duplicated to both
    # formats so each format slice is self-contained)
    shared_metric_map: dict = field(default_factory=lambda: {
        "overall_trend":           "overallTrend",
        "positional_trend":        "positionalTrend",
        "overall_7day_trend":      "overall7DayTrend",
        "positional_7day_trend":   "positional7DayTrend",
        "kept":                    "kept",
        "traded":                  "traded",
        "cut":                     "cut",
        "adp":                     "adp",
        "avg_auction_pct":         "avgAuctionPercentage",
        "startup_adp":             "startupAdp",
        "startup_avg_auction_pct": "startupAvgAuctionPercentage",
        "std_liquidity":           "stdLiquidity",
    })
    fact_rank: str = "fact_dynasty_rankings"
    fact_metrics: str = "fact_dynasty_ranking_metrics"
    crosswalk: str = "dim_dynasty_crosswalk"


KCFG = KTCDynastyConfig()
SNAPSHOT_DATE = date.today().isoformat()
print(f"snapshot_date={SNAPSHOT_DATE} | source={KCFG.source_name}")

In [ ]:
# ---- Scraper ----------------------------------------------------------------
class KTCDynastyScraper:
    """Parse `var playersArray` from the KTC dynasty (overall) page.

    One GET yields the full ~500-asset universe. We emit two format rows per real
    player (SF + TEPP); RDP draft-pick rows are dropped (no player identity).
    """

    _PLAYERS_PAT = re.compile(r"var playersArray\s*=\s*(\[.*?\]);", re.DOTALL)
    _HEADERS = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    def __init__(self, cfg: KTCDynastyConfig = KCFG, timeout: int = 30, delay: float = 2.0):
        self.cfg = cfg
        self.timeout = timeout
        self.delay = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def fetch_players(self) -> list:
        """GET the page, extract + return the playersArray list."""
        time.sleep(self.delay)
        resp = self.session.get(self.cfg.url, timeout=self.timeout)
        resp.raise_for_status()
        m = self._PLAYERS_PAT.search(resp.text)
        if not m:
            raise ValueError("var playersArray not found in KTC dynasty HTML")
        return json.loads(m.group(1))

    def build_rows(self, players: list) -> list[dict]:
        """Flatten players -> long records: one per (player, format), each carrying
        backbone fields + a `_metrics` dict. RDP rows skipped."""
        rows = []
        for p in players:
            if p.get("position") == "RDP":          # rookie draft pick asset, no identity
                continue
            sv = p.get("superflexValues", {})
            if not sv:
                continue
            shared = {k: sv.get(src) for k, src in self.cfg.shared_metric_map.items()}
            ident = {
                "source_player_id": str(p.get("playerID")),
                "player_name":      p.get("playerName"),
                "position_raw":     p.get("position"),
                "nfl_team":         p.get("team"),
                "age":              p.get("age"),
            }
            for fmt, path in self.cfg.formats.items():
                obj = sv
                for key in path[1:]:                # drill past 'superflexValues'
                    obj = obj.get(key, {})
                if not obj:
                    continue
                metrics = {k: obj.get(src) for k, src in self.cfg.fmt_metric_map.items()}
                metrics.update(shared)
                rows.append({
                    **ident,
                    "format":          fmt,
                    "overall_rank":    obj.get("rank"),
                    "positional_rank": obj.get("positionalRank"),
                    "_metrics":        metrics,
                })
        return rows


In [ ]:
# ---- Scrape -----------------------------------------------------------------
scraper = KTCDynastyScraper()
players = scraper.fetch_players()

raw_path = Path(KCFG.__dict__.get("raw_dir", "data/raw")) / f"ktc_dynasty_{SNAPSHOT_DATE}.json"
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw_path.write_text(json.dumps(players, indent=2), encoding="utf-8")

rows = scraper.build_rows(players)
n_rdp = sum(1 for p in players if p.get("position") == "RDP")
print(f"[ok] {len(players)} assets ({n_rdp} RDP skipped) -> {len(rows)} rows "
      f"({len(rows)//2} players x {len(KCFG.formats)} formats) -> raw: {raw_path}")


In [ ]:
# ---- Identity: resolve via shared etl_helpers matcher (single source of truth) --
# Manual gsis overrides for KTC ids that fuzz below threshold (nickname vets):
KTC_OVERRIDES = {
    "533":  "00-0036196",   # Gabriel Davis -> Gabe Davis (WR, BUF)
    "1320": "00-0037809",   # Chigoziem Okonkwo -> Chig Okonkwo (TE)
}                           # Le'Veon Moss (1941): unsigned rookie -> player_key, no gsis
ident = (pd.DataFrame(rows)[["source_player_id", "player_name", "position_raw", "nfl_team"]]
           .drop_duplicates("source_player_id"))
ident["source"] = KCFG.source_name
ktc_xwalk = etl.resolve_dynasty_crosswalk(ident, data_dir=CFG.data_dir, overrides=KTC_OVERRIDES)
print(ktc_xwalk["match_method"].value_counts().to_string())

# Upsert into the unified crosswalk: replace this source's rows, keep others.
xpath = f"{CFG.data_dir}/{KCFG.crosswalk}.parquet"
if Path(xpath).exists():
    prior = pd.read_parquet(xpath)
    prior = prior[prior["source"] != KCFG.source_name]
    xwalk = pd.concat([prior, ktc_xwalk], ignore_index=True)
else:
    xwalk = ktc_xwalk
xwalk.to_parquet(xpath, index=False)

# Review staging only for genuinely unresolved rows (review/unmatched).
rvp = Path(CFG.data_dir) / "review" / "review_dynasty_crosswalk.csv"
review = ktc_xwalk[ktc_xwalk["match_method"].isin(["review", "unmatched"])]
rvp.parent.mkdir(parents=True, exist_ok=True)
if len(review):
    rv = review.copy(); rv["action"] = ""          # user fills: a gsis_id, or "new"
    rv.to_csv(rvp, index=False)
    print(f"[review] {len(review)} -> {rvp}")
else:
    if rvp.exists():
        rvp.unlink()                               # clear stale review once all resolve
    print("[review] none -- all KTC ids resolved")
print(f"[ok] crosswalk: {ktc_xwalk['gsis_id'].notna().sum()}/{len(ktc_xwalk)} gsis "
      f"({ktc_xwalk['gsis_id'].notna().mean()*100:.1f}%) -> {xpath}")


In [ ]:
# ---- Build the two facts + load (replace-by-(snapshot_date, source_name)) ----
fk = xwalk[xwalk["source"] == KCFG.source_name].set_index("source_player_id")
df = pd.DataFrame(rows)
df["snapshot_date"] = SNAPSHOT_DATE
df["source_name"]   = KCFG.source_name
df["source_site"]   = KCFG.source_site
df["gsis_id"]       = df["source_player_id"].map(fk["gsis_id"])
df["player_key"]    = df["source_player_id"].map(fk["player_key"])

backbone_cols = ["snapshot_date", "source_name", "source_player_id", "format",
                 "source_site", "player_name", "position_raw", "nfl_team", "age",
                 "overall_rank", "positional_rank", "gsis_id", "player_key"]
backbone = df[backbone_cols].copy()

# metrics: explode the _metrics dict to long; numeric -> metric_num.
# Iterate the row dicts directly (itertuples mangles leading-underscore columns).
mrecs = []
for r in rows:
    for mk, mv in r["_metrics"].items():
        if mv is None:
            continue
        is_num = isinstance(mv, (int, float)) and not isinstance(mv, bool)
        mrecs.append({
            "snapshot_date": SNAPSHOT_DATE, "source_name": KCFG.source_name,
            "source_player_id": r["source_player_id"], "format": r["format"],
            "metric_key": mk,
            "metric_num": float(mv) if is_num else None,
            "metric_text": None if is_num else str(mv),
        })
metrics = pd.DataFrame.from_records(mrecs)

def load_partition(new: pd.DataFrame, name: str, parts: list[str]) -> str:
    """Replace-by-partition append: drop existing rows for the (parts) values in
    `new`, then concat. Idempotent re-runs for the same snapshot/source."""
    path = f"{CFG.data_dir}/{name}.parquet"
    if Path(path).exists():
        old = pd.read_parquet(path)
        keys = set(map(tuple, new[parts].drop_duplicates().to_numpy()))
        mask = old[parts].apply(tuple, axis=1).isin(keys)
        out = pd.concat([old[~mask], new], ignore_index=True)
    else:
        out = new
    out.to_parquet(path, index=False)
    return path

# Single-column surrogate for PBI relationships (source_player_id alone is not unique).
backbone["source_uid"] = backbone["source_name"] + "|" + backbone["source_player_id"]
metrics["source_uid"]  = metrics["source_name"] + "|" + metrics["source_player_id"]
bp = load_partition(backbone, KCFG.fact_rank,    ["snapshot_date", "source_name"])
mp = load_partition(metrics,  KCFG.fact_metrics, ["snapshot_date", "source_name"])
print(f"[ok] {len(backbone)} backbone rows -> {bp}")
print(f"[ok] {len(metrics)} metric rows -> {mp}")


In [ ]:
# ---- Validation -------------------------------------------------------------
bb = pd.read_parquet(f"{CFG.data_dir}/{KCFG.fact_rank}.parquet")
mt = pd.read_parquet(f"{CFG.data_dir}/{KCFG.fact_metrics}.parquet")

print("=== fact_dynasty_rankings ===")
print(f"rows: {len(bb)} | this snapshot: {(bb['snapshot_date']==SNAPSHOT_DATE).sum()}")
print(f"gsis coverage: {bb['gsis_id'].notna().mean()*100:.1f}% | "
      f"player_key: {bb['player_key'].notna().mean()*100:.1f}%")
print("by format:"); print(bb[bb['snapshot_date']==SNAPSHOT_DATE]
      .groupby('format').size().to_string())
dups = bb.duplicated(subset=["snapshot_date","source_name","source_player_id","format"]).sum()
print(f"backbone key dupes: {dups} (expect 0)")

print("\n=== fact_dynasty_ranking_metrics ===")
print(f"rows: {len(mt)} | distinct metric_key: {mt['metric_key'].nunique()}")
print(mt.groupby('metric_key').size().to_string())
mdups = mt.duplicated(subset=["snapshot_date","source_name","source_player_id","format","metric_key"]).sum()
print(f"metric key dupes: {mdups} (expect 0)")

print("\n=== top 10 SF by overall_rank ===")
top = bb[(bb['snapshot_date']==SNAPSHOT_DATE) & (bb['format']=='SF')].sort_values('overall_rank')
print(top[['overall_rank','positional_rank','player_name','position_raw','age','gsis_id']]
      .head(10).to_string(index=False))
